In [1]:
# 2.1 Import Library dan Konfigurasi Path
import pandas as pd
import numpy as np
import os
from collections import Counter

# Konfigurasi path
DATA_PATH = '../../outputs/evaluation/ground_truth_final.csv'
LEXICON_PATH = '../../../kamus/inset_plus_politik.csv'
OUTPUT_DIR = '../../outputs/RSN'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("[INFO] Library dan konfigurasi path berhasil dimuat.")

[INFO] Library dan konfigurasi path berhasil dimuat.


In [2]:
# 2.2 Load Data Ground Truth
df = pd.read_csv(DATA_PATH)

print(f"\n[DATA] Ground truth berhasil dimuat: {len(df)} tweet")
print(f"Kolom tersedia: {df.columns.tolist()}")
df.head()


[DATA] Ground truth berhasil dimuat: 450 tweet
Kolom tersedia: ['no', 'timestamp', 'teks', 'teks_processed', 'ground_truth_mean', 'ground_truth_scaled', 'sd_annotator', 'ground_truth_category']


,no,timestamp,teks,teks_processed,ground_truth_mean,ground_truth_scaled,sd_annotator,ground_truth_category
0,50,2016-12-24T13:54:06.000Z,Komisi IX Ketua Komisi IX DPR RI merekomendasi...,komisi IX ketua komisi IX DPR RI rekomendasi k...,0.8,0.20,0.447214,positive
1,84,2016-12-21T08:47:51.000Z,Dapilku Rumahku Reses dan silaturahim bersama ...,dapilku rumah reses dan silaturahim sama anggo...,0.8,0.20,0.447214,positive
2,89,2016-12-20T18:32:07.000Z,mengharukan wakil rakyat trlalu banyak waktuny...,haru wakil rakyat terlalu banyak waktu untuk u...,-2.8,-0.70,0.447214,negative
3,101,2016-12-20T04:39:14.000Z,Ketua dan Para Wakil Ketua DPR RI PERLUNYA RUU...,ketua dan para wakil ketua DPR RI PERLU RUU LA...,0.8,0.20,0.447214,positive
4,121,2016-12-18T04:07:09.000Z,Viva Anggota DPR Minta TNI AU Investigasi Kece...,viva anggota DPR minta TNI AU investigasi cela...,0.6,0.15,0.547723,positive


In [3]:
# 2.3 Load Leksikon Gabungan

df_inset = pd.read_csv(LEXICON_PATH)       
df_inset['kata'] = df_inset['kata'].astype(str).str.strip().str.lower()

# Dictionary lexicon untuk matching
inset_dict = dict(zip(df_inset['kata'], df_inset['skor'])) 

print(f"[LEKSIKON] InSet+Politik berhasil dimuat: {len(inset_dict)} entri.")

[LEKSIKON] InSet+Politik berhasil dimuat: 9855 entri.


In [4]:
# 2.4 Definisi Kategori Kata Fungsi (Untuk Ignore Function)
NEGASI_DAN_MODAL = {
    'tidak', 'bukan', 'jangan', 'belum', 'sangat', 'harus', 'wajib',
    'akan', 'sudah', 'sedang', 'telah', 'boleh', 'bisa'
}

KATA_HUBUNG_PREPOSISI = {
    'dan', 'atau', 'tetapi', 'karena', 'jika', 'di', 'ke', 'dari',
    'pada', 'untuk', 'dengan', 'oleh', 'hingga', 'sejak'
}

PRONOMINA_DEMONSTRATIVA = {
    'saya', 'aku', 'dia', 'kami', 'kamu', 'anda', 'ini', 'itu', 'yang'
}

PARTIKEL_KATA_TANYA = {
    'pun', 'sih', 'ya', 'lah', 'kah', 'apa', 'siapa', 'bagaimana'
}

ALL_FUNCTION_WORDS = (
    NEGASI_DAN_MODAL
    | KATA_HUBUNG_PREPOSISI
    | PRONOMINA_DEMONSTRATIVA
    | PARTIKEL_KATA_TANYA
)

In [5]:
# 2.5 Konfigurasi Remove Set 
remove_set_final = set(df_inset[df_inset['kata'].isin(ALL_FUNCTION_WORDS)]['kata'])

print(f"\n[KONFIGURASI REMOVE FUNCTION]")
print(f"Total kata fungsi dalam definisi: {len(ALL_FUNCTION_WORDS)} kata")
print(f"Kata fungsi yang akan DIHAPUS fisik dari teks: {len(remove_set_final)} kata")


# 2.6 Fungsi Tokenisasi
def tokenize(text):
    """Tokenisasi sederhana berdasarkan spasi"""
    if not isinstance(text, str):
        return []
    return text.split()

df['tokens'] = df['teks_processed'].apply(tokenize)

total_tokens = df['tokens'].str.len().sum()
print(f"\n[TOKENISASI] Selesai. Total token: {total_tokens:,}")


[KONFIGURASI REMOVE FUNCTION]
Total kata fungsi dalam definisi: 44 kata
Kata fungsi yang akan DIHAPUS fisik dari teks: 22 kata

[TOKENISASI] Selesai. Total token: 8,282


In [6]:
# 2.6 Fungsi Tokenisasi
def tokenize(text):
    if not isinstance(text, str):
        return []
    return text.split()

df['tokens'] = df['teks_processed'].apply(tokenize)
print(f"\n[INFO] Tokenisasi selesai. Total token: {df['tokens'].str.len().sum():,}")


[INFO] Tokenisasi selesai. Total token: 8,282


In [7]:
# 2.7 Fungsi Lexicon Matching dengan Penghapusan Kata Fungsi
def match_lexicon_remove(tokens, lexicon, remove_set):
    """
    Memisahkan token menjadi 2 kategori (Kata fungsi dihapus):
    1. Matched: Kata yang ada di leksikon (bukan fungsi)
    2. Unmatched: Kata yang tidak ada di leksikon
    Note: Kata fungsi tidak dikembalikan sama sekali.
    """
    matched_words = []
    unmatched_words = []
    
    for token in tokens:
        token_lower = token.lower()
        
        # Cek apakah kata fungsi -> HAPUS
        if token_lower in remove_set:
            continue 
            
        # Cek apakah ada di leksikon -> MATCH
        elif token_lower in lexicon:
            matched_words.append(token)
            
        # Sisanya -> UNMATCH
        else:
            unmatched_words.append(token)
            
    return matched_words, unmatched_words

In [8]:
# 2.8 Penerapan Lexicon Matching
print("\n[PROSES] Menjalankan lexicon matching InSet+Politik (Remove Function)...")

# Output hanya 2 kolom: matched dan unmatched
df[['matched_words', 'unmatched_words']] = pd.DataFrame(
    df['tokens'].apply(
        lambda x: match_lexicon_remove(x, inset_dict, remove_set_final)
    ).tolist(),
    index=df.index
)

print("[INFO] Lexicon matching selesai.")


[PROSES] Menjalankan lexicon matching InSet+Politik (Remove Function)...
[INFO] Lexicon matching selesai.


In [9]:
# 2.9 Perhitungan Statistik
total_words = df['tokens'].str.len().sum()
total_matched = df['matched_words'].str.len().sum()
total_unmatched = df['unmatched_words'].str.len().sum()
total_removed = total_words - (total_matched + total_unmatched)

print("\n[STATISTIK] Hasil Lexicon Matching (RSN + Remove Function):")
print(f"Total kata             : {total_words:,}")
print(f"Matched di InSet+Politik : {total_matched:,} ({(total_matched/total_words)*100:.2f}%)")
print(f"Removed (fungsi)       : {total_removed:,} ({(total_removed/total_words)*100:.2f}%)")
print(f"Unmatched              : {total_unmatched:,} ({(total_unmatched/total_words)*100:.2f}%)")
print("-"*60)
print(f"Coverage Rate          : {((total_matched + total_removed)/total_words)*100:.2f}%")
print(f"  (Matched + Removed)")


[STATISTIK] Hasil Lexicon Matching (RSN + Remove Function):
Total kata             : 8,282
Matched di InSet+Politik : 5,295 (63.93%)
Removed (fungsi)       : 559 (6.75%)
Unmatched              : 2,428 (29.32%)
------------------------------------------------------------
Coverage Rate          : 70.68%
  (Matched + Removed)


In [10]:
# 2.10 Analisis Top Unmatched Words
print("\n[ANALISIS] Top 50 Kata Unmatched Paling Sering Muncul:")
print("-"*40)

all_unmatched = []
for lst in df['unmatched_words']:  
    all_unmatched.extend([w.lower() for w in lst])

unmatched_freq = Counter(all_unmatched)
top_50_unmatched = unmatched_freq.most_common(50)

print(f"{'Kata':<25} | {'Frekuensi':<10}")
print("-"*40)
for word, freq in top_50_unmatched:
    print(f"{word:<25} | {freq:<10}")


[ANALISIS] Top 50 Kata Unmatched Paling Sering Muncul:
----------------------------------------
Kata                      | Frekuensi 
----------------------------------------
dan                       | 113       
di                        | 98        
?                         | 68        
ini                       | 66        
untuk                     | 60        
dengan                    | 43        
ke                        | 41        
!                         | 39        
jika                      | 27        
bagai                     | 27        
akan                      | 25        
juga                      | 22        
saat                      | 22        
tetapi                    | 17        
laksana                   | 15        
uang                      | 14        
atau                      | 14        
belum                     | 13        
terus                     | 13        
tak                       | 12        
h                         | 12        
sah 

In [11]:
# 2.11 Preview Hasil Matching
print("\n[PREVIEW] 5 Tweet Pertama:")

for i in range(min(5, len(df))):
    print(f"\n[Tweet {i+1}]")
    teks_preview = df['teks_processed'].iloc[i]
    if len(teks_preview) > 80:
        teks_preview = teks_preview[:80] + "..."
    print(f"Original: {teks_preview}")
    print(f"✓ Matched : {df['matched_words'].iloc[i]}")
    print(f"✗ Unmatched: {df['unmatched_words'].iloc[i][:5]}...") 


[PREVIEW] 5 Tweet Pertama:

[Tweet 1]
Original: komisi IX ketua komisi IX DPR RI rekomendasi kepada pem agar tinjau ulang kebija...
✓ Matched : ['komisi', 'IX', 'ketua', 'komisi', 'IX', 'DPR', 'RI', 'rekomendasi', 'kepada', 'agar', 'tinjau', 'ulang', 'kebijakan', 'bebas', 'visa']
✗ Unmatched: ['pem']...

[Tweet 2]
Original: dapilku rumah reses dan silaturahim sama anggota DPR RI pkbmembelarakyat
✓ Matched : ['rumah', 'reses', 'silaturahim', 'sama', 'anggota', 'DPR', 'RI', 'pkbmembelarakyat']
✗ Unmatched: ['dapilku', 'dan']...

[Tweet 3]
Original: haru wakil rakyat terlalu banyak waktu untuk update disosmed hampir tidak ada wa...
✓ Matched : ['wakil', 'rakyat', 'terlalu', 'banyak', 'waktu', 'update', 'ada', 'waktu', 'update', 'aspirasi', 'rakyat', 'wakil']
✗ Unmatched: ['haru', 'untuk', 'disosmed', 'hampir', 'untuk']...

[Tweet 4]
Original: ketua dan para wakil ketua DPR RI PERLU RUU LAPOR UANG UNTUK DUNIA BISNIS USAHA ...
✓ Matched : ['ketua', 'para', 'wakil', 'ketua', 'DPR', 'RI', 'P

In [12]:
# 2.12 Simpan Output Matching
output_path = os.path.join(OUTPUT_DIR, 'rsn_lexicon_matching_remove.csv')
df.to_csv(output_path, index=False)

print(f"\n[OUTPUT] Data berhasil disimpan ke: {output_path}")

print("\n[CATATAN PENTING UNTUK NOTEBOOK 3 (SCORING)]")
print("Karena kata fungsi DIHAPUS, maka pada tahap scoring nanti:")
print("Hanya iterasi 'matched_words' yang akan dihitung skornya.")
print("Kata fungsi tidak akan ada sama sekali dalam proses VADER.")


[OUTPUT] Data berhasil disimpan ke: ../../outputs/RSN\rsn_lexicon_matching_remove.csv

[CATATAN PENTING UNTUK NOTEBOOK 3 (SCORING)]
Karena kata fungsi DIHAPUS, maka pada tahap scoring nanti:
Hanya iterasi 'matched_words' yang akan dihitung skornya.
Kata fungsi tidak akan ada sama sekali dalam proses VADER.
